# pgvector with SQLAlchemy: playground

Companion to `docs/sqlalchemy pgvector guide.md` and to the JSONB notebook next door. Every query
comes back as a pandas DataFrame, and the SQL that SQLAlchemy sends is printed above it.

Two tables are used:

- **`toy_items`**: a dozen hand-made 3-dimensional vectors whose axes *mean* something
  (`[fruit, tech, sport]`), so you can predict every distance in your head and check the
  database against numpy.
- **`demo_docs`**: 20,000 random 64-dimensional vectors, for the indexing section, where the
  interesting questions are speed and recall rather than the individual numbers.

Neither touches the project's `documents` table.

**Which server?** The next cell is a [papermill](https://papermill.readthedocs.io/) `parameters`
cell. `SERVER` picks the target: `"pg16"` is the project's Postgres 16 + pgvector image,
`"pg18"` the Postgres 18 image that adds pg_vectorsearch's PRISM index (section 6),
`"pgrust"` is Postgres 18 rewritten in Rust with pgvector bundled (see the README), and
`"k8s-pg16"` / `"k8s-pgrust"` are the same two images running as pods on a local
[kind](https://kind.sigs.k8s.io/) cluster, and `"k3d-pg16"` / `"k3d-pgrust"` on a
[k3d](https://k3d.io/) cluster (both under `k8s/`). The setup cell uses the Docker SDK to find the
container (for kind, the node container), start it if it is stopped, wait until the database
answers, and point `DATABASE_URL` at it. Edit the value and re-run, or run the whole notebook
headless:

```bash
uv run papermill notebooks/pgvector_playground.ipynb out/pgvector_pgrust.ipynb -p SERVER pgrust
```

**Prerequisites**: Docker is running, the container for the chosen server has been created at least
once (`docker compose -f docker/<file>.yaml up -d`), and the kernel is this project's `.venv`
(`uv sync` created it; pick it in the kernel selector). The notebook uses top-level `await`, which
Jupyter supports out of the box.

In [ ]:
# Parameters. Papermill overrides these with `-p NAME value`; interactively, just edit and re-run.
SERVER = "pg16"  # "pg16" / "pg18" / "pgrust" (compose), "k8s-pg16" / "k8s-pgrust" (kind), "k3d-pg16" / "k3d-pgrust" (k3d), "k3s-pg16" / "k3s-pgrust" (k3s via the Docker SDK, started here if needed); see k8s/
START_CONTAINER = True  # start the container with the Docker SDK if it exists but is stopped
N_DOCS = 20_000  # rows in the indexing section
DIM = 64  # their dimension
CONCURRENCY_LEVELS = [1, 4, 16, 32]  # connections at once in section 7 (papermill: -y "CONCURRENCY_LEVELS: [1, 8]")
BENCH_SECONDS = 2.0  # how long each (workload, level) cell runs

In [ ]:
# Locate the project root (works whether the kernel's cwd is the project root or this folder),
# put src/ on the path and load .env so the connection URLs are set.
import os
import pathlib
import sys
import time

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

# Resolve SERVER to a Docker container and a URL variable in .env. The compose servers are the
# containers from docker/*.yaml; the k8s ones live as pods inside the kind node container
# (k8s/kind-cluster.yaml), reachable through the NodePorts kind maps onto the host.
import docker
import psycopg

SERVERS = {
    "pg16": {"container": "pgvector-dev", "url_var": "DATABASE_URL",
             "create": "docker compose -f docker/postgres-pgvector.yaml up -d"},
    "pg18": {"container": "pg18-dev", "url_var": "PG18_DATABASE_URL",
             "create": "docker compose -f docker/pg18.yaml up -d --build"},
    "pgrust": {"container": "pgrust-dev", "url_var": "PGRUST_DATABASE_URL",
               "create": "docker compose -f docker/pgrust.yaml up -d"},
    "k8s-pg16": {"container": "pgbench-control-plane", "url_var": "K8S_PG16_DATABASE_URL",
                 "create": "kind create cluster --config k8s/kind-cluster.yaml  (then see k8s/kind-cluster.yaml)"},
    "k8s-pgrust": {"container": "pgbench-control-plane", "url_var": "K8S_PGRUST_DATABASE_URL",
                   "create": "kind create cluster --config k8s/kind-cluster.yaml  (then see k8s/kind-cluster.yaml)"},
    # k3d publishes host ports on its load-balancer container, not on the node
    "k3d-pg16": {"container": "k3d-pgbench-serverlb", "url_var": "K3D_PG16_DATABASE_URL",
                 "create": "k3d cluster create --config k8s/k3d-cluster.yaml  (then see k8s/k3d-cluster.yaml)"},
    "k3d-pgrust": {"container": "k3d-pgbench-serverlb", "url_var": "K3D_PGRUST_DATABASE_URL",
                   "create": "k3d cluster create --config k8s/k3d-cluster.yaml  (then see k8s/k3d-cluster.yaml)"},
    # k3s in one container, started from here with the same Docker SDK (k8s/k3s.py): no CLI tools needed
    "k3s-pg16": {"container": "pgbench-k3s", "url_var": "K3S_PG16_DATABASE_URL",
                 "create": "uv run k8s/k3s.py up --server pg16"},
    "k3s-pgrust": {"container": "pgbench-k3s", "url_var": "K3S_PGRUST_DATABASE_URL",
                   "create": "uv run k8s/k3s.py up --server pgrust"},
}
cfg = SERVERS[SERVER]

client = docker.from_env()
try:
    ctr = client.containers.get(cfg["container"])
except docker.errors.NotFound:
    if not (SERVER.startswith("k3s-") and START_CONTAINER):
        raise RuntimeError(f"container {cfg['container']!r} does not exist; create it once with\n  {cfg['create']}") from None
    sys.path.insert(0, str(ROOT / "k8s"))
    import k3s

    k3s.up([SERVER.removeprefix("k3s-")])  # cluster + image + manifest + wait, all through docker-py
    ctr = client.containers.get(cfg["container"])

if ctr.status != "running":
    if not START_CONTAINER:
        raise RuntimeError(f"{ctr.name} is {ctr.status} and START_CONTAINER is False")
    print(f"starting {ctr.name} ...")
    ctr.start()

# Wait until the database answers, not just until the container runs (a kind node also has to
# schedule the pod). psycopg wants the plain URL, without SQLAlchemy's "+psycopg" driver tag.
url = os.environ[cfg["url_var"]]
dsn = url.replace("postgresql+psycopg://", "postgresql://")
for attempt in range(90):
    try:
        with psycopg.connect(dsn, connect_timeout=2) as c:
            server = c.execute("SELECT version()").fetchone()[0]
        break
    except psycopg.OperationalError:
        time.sleep(1)
else:
    raise RuntimeError(f"no answer from {dsn.split('@')[-1]} after 90s; check the container/pod logs")

print(f"{SERVER}: container={ctr.name} ({ctr.image.tags[0]}, {ctr.status}), "
      f"db at {dsn.split('@')[-1]} after {attempt}s")
print(server)

# db.py reads DATABASE_URL when it is imported, so set it *before* the import in the next cell.
os.environ["DATABASE_URL"] = url

In [ ]:
import numpy as np
import pandas as pd
from pgvector.sqlalchemy import Vector
import asyncio
import random

from sqlalchemy import Index, Integer, String, bindparam, func, insert, select, text
from sqlalchemy.dialects import postgresql
from sqlalchemy.dialects.postgresql import insert as pg_insert
from sqlalchemy.ext.asyncio import create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# Local modules
from db import SessionLocal, engine

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)
np.set_printoptions(precision=3, suppress=True)

DIALECT = postgresql.psycopg.dialect()


def sql(stmt) -> None:
    '''Print the SQL SQLAlchemy will send, plus bound parameters (vectors shortened).'''
    c = stmt.compile(dialect=DIALECT)
    print(str(c).strip())
    if c.params:
        short = {
            k: (f"<vector[{len(v)}]>" if isinstance(v, (list, np.ndarray)) and len(v) > 8 else v)
            for k, v in c.params.items()
        }
        print("-- params:", short)


async def q(stmt, show=True) -> pd.DataFrame:
    '''Execute a statement in the shared session and return the rows as a DataFrame.'''
    if show:
        sql(stmt)
    result = await session.execute(stmt)
    return pd.DataFrame(result.mappings().all())


async def explain(stmt) -> None:
    '''EXPLAIN ANALYZE a SQLAlchemy statement (binds rendered inline, so no params to pass).'''
    compiled = stmt.compile(dialect=DIALECT, compile_kwargs={"literal_binds": True})
    plan = await session.execute(text(f"EXPLAIN (ANALYZE, COSTS OFF) {compiled}"))
    for (line,) in plan:
        print(line)


session = SessionLocal()
# A fresh database (e.g. a new pod) has no extension yet; same statement init_db() in db.py runs.
async with engine.begin() as conn:
    await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
print("ready:", engine.url.render_as_string(hide_password=True))
print("pgvector", await session.scalar(text("SELECT extversion FROM pg_extension WHERE extname = 'vector'")))

## 0. The data

A `Vector(3)` column. On the Python side you hand it a list (or numpy array) and get a plain
`list[float]` back; wrap it in `np.array(...)` when you want to do maths on it. The three axes are `[fruit, tech, sport]`, so `apple` is mostly fruit, `laptop`
is mostly tech and `smartwatch` sits between tech and sport.

The models live on their own `DeclarativeBase` so re-running this cell doesn't clash with the
project's `Base`. `reset_toys()` recreates the table and reloads the rows; call it whenever you
want a clean slate.

In [ ]:
class ToyBase(DeclarativeBase):
    pass


class Toy(ToyBase):
    __tablename__ = "toy_items"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50), unique=True)
    category: Mapped[str] = mapped_column(String(20))
    popularity: Mapped[int] = mapped_column(Integer)
    # [fruit, tech, sport]; nullable so we can see what a NULL vector does to a search
    embedding: Mapped[list[float] | None] = mapped_column(Vector(3), nullable=True)


TOYS = [
    # name          category  pop  embedding
    ("apple",       "fruit",  90, [0.90, 0.10, 0.00]),
    ("banana",      "fruit",  70, [0.80, 0.00, 0.10]),
    ("cherry",      "fruit",  40, [0.95, 0.05, 0.05]),
    ("giant apple", "fruit",  10, [2.70, 0.30, 0.00]),  # same direction as apple, 3x longer
    ("laptop",      "tech",   85, [0.00, 0.90, 0.10]),
    ("phone",       "tech",   95, [0.10, 0.85, 0.05]),
    ("keyboard",    "tech",   50, [0.05, 0.80, 0.00]),
    ("smartwatch",  "tech",   60, [0.00, 0.60, 0.60]),
    ("football",    "sport",  80, [0.00, 0.10, 0.90]),
    ("tennis",      "sport",  55, [0.10, 0.00, 0.85]),
    ("skates",      "sport",  30, [0.00, 0.20, 0.80]),
    ("mystery",     "other",   5, None),                 # no embedding yet
]


async def reset_toys() -> None:
    # The session keeps a transaction open between cells (and with it, locks on the tables it
    # has read). DDL runs on a different connection, so end that transaction first or the
    # DROP TABLE below waits forever.
    await session.rollback()
    async with engine.begin() as conn:
        await conn.run_sync(ToyBase.metadata.drop_all, tables=[Toy.__table__])
        await conn.run_sync(ToyBase.metadata.create_all, tables=[Toy.__table__])
    session.add_all(
        Toy(name=n, category=c, popularity=p, embedding=e) for n, c, p, e in TOYS
    )
    await session.commit()


await reset_toys()
toys = await q(select(Toy.id, Toy.name, Toy.category, Toy.popularity, Toy.embedding).order_by(Toy.id))
toys

In [ ]:
# What comes back: the pgvector type parses Postgres's '[0.9,0.1,0]' text into a Python list
# (None for NULL). You can insert numpy arrays, but you always read lists.
session.expire_all()  # otherwise the ORM returns the object we added above without re-reading it
row = await session.scalar(select(Toy).where(Toy.name == "apple"))
print(type(row.embedding), row.embedding)
print("as numpy:", np.array(row.embedding))
print("column type:", Toy.__table__.c.embedding.type)

## 1. The four distances

Each operator is a method on the column. All of them take a Python list or numpy array on the
right-hand side and return a float.

| SQL | SQLAlchemy | What it is | Index opclass |
|---|---|---|---|
| `a <-> b` | `col.l2_distance(b)` | Euclidean distance, `sqrt(sum((a-b)^2))` | `vector_l2_ops` |
| `a <=> b` | `col.cosine_distance(b)` | `1 - cos(angle)`: 0 same direction, 1 orthogonal, 2 opposite | `vector_cosine_ops` |
| `a <#> b` | `col.max_inner_product(b)` | **negative** dot product, `-(a . b)` | `vector_ip_ops` |
| `a <+> b` | `col.l1_distance(b)` | Manhattan distance, `sum(abs(a-b))` | `vector_l1_ops` |

Why is the inner product negated? Postgres can only use an index for `ORDER BY x ASC`
(smallest first), and for inner product you want the *largest* value. Flipping the sign makes
"best match first" and "smallest first" the same thing. Negate it again if you want the real
dot product in your results.

The query below asks: which items are close to `[0.7, 0.2, 0.1]`, "mostly fruit, a bit techy"?

In [ ]:
qv = [0.7, 0.2, 0.1]

dist_cols = [
    Toy.embedding.l2_distance(qv).label("l2"),
    Toy.embedding.cosine_distance(qv).label("cosine"),
    Toy.embedding.max_inner_product(qv).label("neg_inner"),
    Toy.embedding.l1_distance(qv).label("l1"),
]

db_dist = await q(
    select(Toy.name, Toy.embedding, *dist_cols)
    .where(Toy.embedding.is_not(None))
    .order_by("cosine")
)
db_dist

Look at `giant apple`: by cosine it is as close to the query as `apple` (same direction), by L2 it
is the *farthest* item in the table (it is long), and by inner product it is the best match of all
(long vectors win dot products). That is the whole reason the choice of metric matters:

- **Cosine** ignores magnitude. Use it when only direction carries meaning, which is the case for
  most text-embedding models.
- **L2** cares about magnitude and position. Natural for coordinates, image features, anything
  where "how far" is literal.
- **Inner product** is cosine *without* the normalisation. If your vectors are already unit
  length (OpenAI embeddings are), it ranks identically to cosine and is a little cheaper.

Now the same numbers in numpy, to convince yourself the database isn't doing anything magic.

In [ ]:
E = np.array(db_dist["embedding"].tolist())  # (n, 3) matrix of the stored vectors
qa = np.array(qv)

np_dist = pd.DataFrame(
    {
        "name": db_dist["name"],
        "l2": np.linalg.norm(E - qa, axis=1),
        "cosine": 1 - (E @ qa) / (np.linalg.norm(E, axis=1) * np.linalg.norm(qa)),
        "neg_inner": -(E @ qa),
        "l1": np.abs(E - qa).sum(axis=1),
    }
)
assert np.allclose(np_dist[["l2", "cosine", "neg_inner", "l1"]], db_dist[["l2", "cosine", "neg_inner", "l1"]], atol=1e-6)
print("numpy and Postgres agree to 1e-6")
np_dist

## 2. Nearest-neighbour search

The core pattern is `ORDER BY distance LIMIT k`. Everything else is decoration:

- label the distance and select it so you can see the score;
- put ordinary `WHERE` clauses in front (Postgres filters first, then ranks);
- add `.where(distance < threshold)` for "only good enough matches".

In [ ]:
distance = Toy.embedding.cosine_distance(qv).label("distance")

await q(select(Toy.name, Toy.category, distance).order_by(distance).limit(3))

In [ ]:
# Filter first, then rank: nearest *tech* items to a fruity query
await q(
    select(Toy.name, Toy.category, distance)
    .where(Toy.category == "tech")
    .order_by(distance)
    .limit(3)
)

In [ ]:
# Threshold instead of top-k. The label can't be reused in WHERE (WHERE runs before SELECT),
# so repeat the expression; SQLAlchemy renders it twice.
await q(
    select(Toy.name, distance)
    .where(Toy.embedding.cosine_distance(qv) < 0.1)
    .order_by(distance)
)

**The NULL trap.** A NULL vector has a NULL distance. `ORDER BY ... ASC` puts NULLs last, so with a
`LIMIT` they usually stay out of sight, but without one, or with `DESC`, they show up. And a
threshold filter drops them silently, just like the JSONB missing-key case. Say what you mean:
`.where(Toy.embedding.is_not(None))`.

In [ ]:
print((await q(select(Toy.name, distance).order_by(distance), show=False)).tail(3), "\n")
print("is_not(None):",
      (await q(select(Toy.name).where(Toy.embedding.is_not(None)), show=False))["name"].size,
      "of", len(TOYS), "rows have a vector")

## 3. Vector functions and arithmetic

pgvector ships functions and operators beyond the four distances. SQLAlchemy has no wrappers for
them, so use `func.<name>(...)` and `.op("+")`. Two things to know:

- **Tell SQLAlchemy the return type** when the result is a vector: `func.l2_normalize(col, type_=Vector(3))`.
  Without it the driver returns the raw text `'[0.1,0.2,0.3]'` instead of a parsed list
  (compare the two columns below).
- `vector_dims`, `vector_norm` return plain numbers and need nothing.

| SQL | SQLAlchemy |
|---|---|
| `vector_dims(v)` | `func.vector_dims(Toy.embedding)` |
| `vector_norm(v)` | `func.vector_norm(Toy.embedding)` |
| `l2_normalize(v)` | `func.l2_normalize(Toy.embedding, type_=Vector(3))` |
| `subvector(v, 1, 2)` | `func.subvector(Toy.embedding, 1, 2, type_=Vector(2))` (1-based start) |
| `v + w`, `v - w`, `v * w` | `Toy.embedding.op("+")(other)` (element-wise) |
| `avg(v)`, `sum(v)` | `func.avg(Toy.embedding, type_=Vector(3))`, or `pgvector.sqlalchemy.avg(Toy.embedding)` which infers the type |

In [ ]:
await q(
    select(
        Toy.name,
        func.vector_dims(Toy.embedding).label("dims"),
        func.vector_norm(Toy.embedding).label("norm"),
        func.l2_normalize(Toy.embedding, type_=Vector(3)).label("unit"),
        func.l2_normalize(Toy.embedding).label("unit_untyped"),  # comes back as str
        func.subvector(Toy.embedding, 1, 2, type_=Vector(2)).label("first_two"),
    )
    .where(Toy.name.in_(["apple", "giant apple", "smartwatch"]))
    .order_by(Toy.name)
)

**Centroids.** `avg()` over a vector column gives the mean vector of a group. A classic use: find
the most typical member of each category, i.e. the item closest to its own category's centroid.
The centroid query becomes a CTE, and the distance is computed between two *columns*
(`Toy.embedding` and `centroids.c.centroid`), not a column and a Python list.

In [ ]:
centroids = (
    select(Toy.category, func.avg(Toy.embedding, type_=Vector(3)).label("centroid"))
    .where(Toy.embedding.is_not(None))
    .group_by(Toy.category)
    .cte("centroids")
)
to_centroid = Toy.embedding.cosine_distance(centroids.c.centroid).label("to_centroid")

await q(
    select(Toy.category, Toy.name, centroids.c.centroid, to_centroid)
    .join(centroids, centroids.c.category == Toy.category)
    .order_by(Toy.category, to_centroid)
)

In [ ]:
# Arithmetic: "apple minus fruit plus tech" should land near the tech items (word2vec style)
apple = (await session.scalar(select(Toy).where(Toy.name == "apple"))).embedding
fruit = np.array([1.0, 0.0, 0.0])
tech = np.array([0.0, 1.0, 0.0])
target = (apple - fruit + tech).tolist()
print("target:", np.round(target, 2))

await q(
    select(Toy.name, Toy.embedding.cosine_distance(target).label("distance"))
    .where(Toy.embedding.is_not(None))
    .order_by("distance")
    .limit(3)
)

## 4. Retrieve, then rerank: CTEs and window functions

Real systems rarely return raw nearest neighbours. Two common shapes:

1. **Candidates then rerank**: pull the top-N by vector distance into a CTE, then order those by
   a blended score (here: 70% similarity, 30% popularity). Cheap, because the expensive ranking only
   runs over N rows.
2. **Top-k per group** with `row_number() OVER (PARTITION BY ... ORDER BY distance)`.

In [ ]:
candidates = (
    select(Toy.id, Toy.name, Toy.category, Toy.popularity, distance)
    .where(Toy.embedding.is_not(None))
    .order_by(distance)
    .limit(6)
    .cte("candidates")
)
score = (
    0.7 * (1 - candidates.c.distance) + 0.3 * (candidates.c.popularity / 100.0)
).label("score")

await q(
    select(candidates.c.name, candidates.c.distance, candidates.c.popularity, score)
    .order_by(score.desc())
    .limit(3)
)

In [ ]:
# Best match per category
rank = (
    func.row_number()
    .over(partition_by=Toy.category, order_by=Toy.embedding.cosine_distance(qv))
    .label("rank")
)
ranked = (
    select(Toy.name, Toy.category, distance, rank)
    .where(Toy.embedding.is_not(None))
    .subquery("ranked")
)
await q(select(ranked).where(ranked.c.rank <= 1).order_by(ranked.c.distance))

## 5. Upserting embeddings

Embedding pipelines re-run. `INSERT ... ON CONFLICT DO UPDATE` keyed on a unique column
(`name` here) inserts new rows and refreshes existing ones in one statement. `stmt.excluded`
refers to the row that *would* have been inserted.

Below, `mystery` finally gets a vector and `apple` is re-embedded slightly differently.

In [ ]:
upsert = pg_insert(Toy).values(
    [
        {"name": "mystery", "category": "other", "popularity": 5, "embedding": [0.3, 0.3, 0.3]},
        {"name": "apple", "category": "fruit", "popularity": 90, "embedding": [0.92, 0.08, 0.0]},
    ]
)
upsert = upsert.on_conflict_do_update(
    index_elements=[Toy.name],
    set_={"embedding": upsert.excluded.embedding, "popularity": upsert.excluded.popularity},
).returning(Toy.id, Toy.name, Toy.embedding)

result = await q(upsert)
await session.commit()  # q() only executes; without this the upsert is undone by the next rollback()
result

In [ ]:
# mystery now participates in searches, apple is still the top hit
await q(select(Toy.name, distance).order_by(distance).limit(4), show=False)

## 6. Indexing: exact vs approximate

With no index, Postgres computes the distance to *every* row and sorts: exact, and fine up to
some tens of thousands of rows. Beyond that you add an approximate-nearest-neighbour (ANN) index.
HNSW is the one to use: no training pass, handles inserts, best speed/recall trade-off.

The toy table is too small to show anything, so this section uses `demo_docs`: `N_DOCS` random
unit vectors in `DIM` dimensions (20,000 x 64 by default, see the parameters cell), split over
10 categories. Random vectors are the *worst* case
for ANN (no cluster structure), which makes the recall numbers below pessimistic.

In [ ]:
class DemoDoc(ToyBase):
    __tablename__ = "demo_docs"

    id: Mapped[int] = mapped_column(primary_key=True)
    category: Mapped[str] = mapped_column(String(20))
    embedding: Mapped[list[float]] = mapped_column(Vector(DIM))


N = N_DOCS
rng = np.random.default_rng(42)
M = rng.standard_normal((N, DIM)).astype(np.float32)
M /= np.linalg.norm(M, axis=1, keepdims=True)  # unit length, like most embedding models

await session.rollback()  # release the session's locks before DDL on another connection
async with engine.begin() as conn:
    await conn.run_sync(ToyBase.metadata.drop_all, tables=[DemoDoc.__table__])
    await conn.run_sync(ToyBase.metadata.create_all, tables=[DemoDoc.__table__])

t0 = time.perf_counter()
rows = [{"category": f"cat{i % 10}", "embedding": M[i]} for i in range(N)]
for start in range(0, N, 5_000):  # executemany in chunks
    await session.execute(insert(DemoDoc), rows[start : start + 5_000])
await session.commit()
print(f"inserted {N} x {DIM} in {time.perf_counter() - t0:.1f}s")

### Exact search (sequential scan)

`explain()` runs `EXPLAIN ANALYZE` on the same SQLAlchemy statement. Look for `Seq Scan` and the
execution time.

In [ ]:
qdoc = M[0] + 0.1 * rng.standard_normal(DIM)  # a query near document 1
qdoc = (qdoc / np.linalg.norm(qdoc)).tolist()

knn = (
    select(DemoDoc.id, DemoDoc.category, DemoDoc.embedding.cosine_distance(qdoc).label("distance"))
    .order_by("distance")
    .limit(10)
)
sql(knn)
await explain(knn)
exact_top10 = (await q(knn, show=False))["id"].tolist()
print("\nexact top-10:", exact_top10)

### Add an HNSW index

The `Index()` is plain SQLAlchemy with three Postgres-specific arguments (`hnsw.name` is reused below to size and later rebuild it). The opclass **must match
the operator you query with** (`vector_cosine_ops` for `<=>`); an index built for L2 is ignored by
a cosine query.

- `m`: links per node (16 default). More = better recall, more memory.
- `ef_construction`: candidate list size while building (64 default). More = better index, slower build.
- `hnsw.ef_search`: candidate list size while *querying* (40 default). Set per session or
  transaction; this is your recall/speed knob at runtime.

In [ ]:
hnsw = Index(
    "ix_demo_docs_embedding_hnsw",
    DemoDoc.embedding,
    postgresql_using="hnsw",
    postgresql_with={"m": 16, "ef_construction": 64},
    postgresql_ops={"embedding": "vector_cosine_ops"},
)
print(str(__import__("sqlalchemy").schema.CreateIndex(hnsw).compile(dialect=DIALECT)).strip())

t0 = time.perf_counter()
await session.rollback()  # same reason as above: DDL needs the session's locks released
async with engine.begin() as conn:
    await conn.run_sync(hnsw.create, checkfirst=True)
hnsw_build_s = time.perf_counter() - t0
hnsw_size = await session.scalar(select(func.pg_relation_size(hnsw.name)))
print(f"built in {hnsw_build_s:.1f}s, index size {hnsw_size / 1e6:.1f} MB")

await explain(knn)

### Recall vs `ef_search`

ANN means the answer can be *slightly wrong*. Recall@10 is "how many of the true top-10 did the
index return". `SET LOCAL` lasts until the end of the current transaction, which in this notebook
is until the next `commit()` or `rollback()`; in application code set it inside the transaction
that runs the search.

The timings include the round trips for the two `SET`s, and 20,000 rows is small enough that an
exact scan is only a few milliseconds, so read the recall column, not the speedup. The `EXPLAIN`
output above is the fairer speed comparison.

In [ ]:
async def top10(vec, use_index: bool, ef: int = 40) -> set[int]:
    stmt = select(DemoDoc.id).order_by(DemoDoc.embedding.cosine_distance(vec)).limit(10)
    await session.execute(text(f"SET LOCAL enable_indexscan = {'on' if use_index else 'off'}"))
    await session.execute(text(f"SET LOCAL hnsw.ef_search = {ef}"))
    return set((await session.execute(stmt)).scalars())


queries = rng.standard_normal((30, DIM))
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

report = []
for ef in (10, 40, 100, 400):
    hits, t_idx, t_exact = 0, 0.0, 0.0
    for v in queries:
        v = v.tolist()
        t0 = time.perf_counter(); exact = await top10(v, use_index=False); t_exact += time.perf_counter() - t0
        t0 = time.perf_counter(); approx = await top10(v, use_index=True, ef=ef); t_idx += time.perf_counter() - t0
        hits += len(exact & approx)
    report.append({"ef_search": ef, "recall@10": hits / (10 * len(queries)),
                   "ms/query index": 1000 * t_idx / len(queries), "ms/query exact": 1000 * t_exact / len(queries)})
await session.rollback()  # discard the SET LOCALs
hnsw_report = pd.DataFrame(report)
hnsw_report.round(3)

### Filtering with an index

HNSW walks the graph to collect `ef_search` candidates *then* applies your `WHERE`. If the filter
is selective (here `category = 'cat3'` keeps 10% of rows), most candidates are thrown away and you
can get fewer than `LIMIT` rows back. pgvector 0.8 added **iterative scans** that keep walking until
the limit is satisfied: `SET hnsw.iterative_scan = relaxed_order`.

In [ ]:
filtered = (
    select(DemoDoc.id)
    .where(DemoDoc.category == "cat3")
    .order_by(DemoDoc.embedding.cosine_distance(qdoc))
    .limit(10)
)
for mode in ("off", "relaxed_order"):
    await session.execute(text(f"SET LOCAL hnsw.iterative_scan = {mode}"))
    await session.execute(text("SET LOCAL hnsw.ef_search = 10"))  # deliberately small
    n = len((await session.execute(filtered)).all())
    print(f"iterative_scan={mode:<14} rows returned: {n} of 10 requested")
await session.rollback()

### DiskANN: the other graph index

[pgvectorscale](https://github.com/timescale/pgvectorscale) adds a second ANN index type,
`diskann`, an implementation of Microsoft's StreamingDiskANN. Same idea as HNSW (a proximity
graph you walk), different trade-offs:

- **one flat graph instead of layers**, built to stay fast when it does *not* fit in RAM;
- **compressed vectors in the index** (statistical binary quantization, `storage_layout =
  memory_optimized`, the default), so the index is much smaller, plus a **rescore** step that
  re-reads the top candidates' full vectors from the heap to fix the ranking;
- **query knobs**: `diskann.query_search_list_size` (like `ef_search`, default 100) and
  `diskann.query_rescore` (how many candidates to re-rank exactly, default 50).

It is compiled into the project's pg16 and pg18 images (`docker/Dockerfile`, `docker/Dockerfile.pg18`). pgrust has no extension
ABI, so on that server the cell below reports "not available" and the rest of this subsection is
skipped.

In [ ]:
await session.rollback()
try:
    async with engine.begin() as conn:
        await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vectorscale CASCADE"))
    ver = await session.scalar(text("SELECT extversion FROM pg_extension WHERE extname = 'vectorscale'"))
    HAS_DISKANN = True
    print("pgvectorscale", ver)
except Exception as e:  # noqa: BLE001
    HAS_DISKANN = False
    await session.rollback()
    print("diskann not available on this server:", str(e).splitlines()[0])

Build it on the same table. HNSW is dropped first so the planner has exactly one ANN index to pick
and the measurements are not ambiguous; it comes back at the end of the subsection.
`num_neighbors` (default 50) and `search_list_size` (default 100) are the build-time analogues of
HNSW's `m` and `ef_construction`.

In [ ]:
diskann = Index(
    "ix_demo_docs_embedding_diskann",
    DemoDoc.embedding,
    postgresql_using="diskann",
    postgresql_with={"num_neighbors": 50, "search_list_size": 100},
    postgresql_ops={"embedding": "vector_cosine_ops"},
)
if HAS_DISKANN:
    print(str(__import__("sqlalchemy").schema.CreateIndex(diskann).compile(dialect=DIALECT)).strip())
    await session.rollback()
    t0 = time.perf_counter()
    async with engine.begin() as conn:
        await conn.run_sync(hnsw.drop, checkfirst=True)
        await conn.run_sync(diskann.create, checkfirst=True)
    diskann_build_s = time.perf_counter() - t0
    diskann_size = await session.scalar(select(func.pg_relation_size(diskann.name)))
    print(f"built in {diskann_build_s:.1f}s, index size {diskann_size / 1e6:.1f} MB")
    await explain(knn)

In [ ]:
# Recall and latency across the two query knobs, same 30 queries as the HNSW cell
async def top10_diskann(vec, search_list: int, rescore: int) -> set[int]:
    stmt = select(DemoDoc.id).order_by(DemoDoc.embedding.cosine_distance(vec)).limit(10)
    await session.execute(text("SET LOCAL enable_indexscan = on"))  # the exact loop below turns it off
    await session.execute(text(f"SET LOCAL diskann.query_search_list_size = {search_list}"))
    await session.execute(text(f"SET LOCAL diskann.query_rescore = {rescore}"))
    return set((await session.execute(stmt)).scalars())


diskann_report = pd.DataFrame()
if HAS_DISKANN:
    exact_sets = [await top10(v.tolist(), use_index=False) for v in queries]
    await session.rollback()  # SET LOCAL lasts until the transaction ends; start clean for the index runs
    rows = []
    for search_list, rescore in [(20, 0), (50, 50), (100, 50), (200, 100), (400, 200), (400, 400)]:
        hits, t = 0, 0.0
        for v, exact in zip(queries, exact_sets):
            t0 = time.perf_counter()
            approx = await top10_diskann(v.tolist(), search_list, rescore)
            t += time.perf_counter() - t0
            hits += len(exact & approx)
        rows.append({"search_list_size": search_list, "rescore": rescore,
                     "recall@10": hits / (10 * len(queries)), "ms/query": 1000 * t / len(queries)})
    await session.rollback()
    diskann_report = pd.DataFrame(rows)
diskann_report.round(3)

`rescore = 0` shows what the compressed vectors alone can do; the recall jump at `rescore = 50` is
the exact re-ranking doing its job. The comparison that matters is at equal recall: which index
gets there in less time, and at what size on disk.

Expect DiskANN to look bad on *this* data. Binary quantization keeps one bit per dimension (plus
statistics), which is a good approximation for real embeddings (hundreds of correlated dimensions)
and a poor one for 64 independent Gaussian coordinates, so the candidate list it hands to the
rescore step is often wrong and no amount of rescoring recovers the true top-10. A larger graph
(`num_neighbors = 100, search_list_size = 200`) was tried while writing this: build time went from
15 s to 120 s and recall did not move. The `storage_layout = 'plain'` option skips the
quantization, but in pgvectorscale 0.9.1 its build on these 20k rows did not finish within ten
minutes and ignored `pg_cancel_backend`, so it is left out of the notebook. Re-run this subsection
with real embeddings before drawing conclusions.

In [ ]:
if HAS_DISKANN:
    summary = pd.DataFrame(
        [
            {"index": "hnsw", "build s": hnsw_build_s, "size MB": hnsw_size / 1e6,
             "best recall@10": hnsw_report["recall@10"].max(),
             "ms/query at best": hnsw_report.loc[hnsw_report["recall@10"].idxmax(), "ms/query index"]},
            {"index": "diskann", "build s": diskann_build_s, "size MB": diskann_size / 1e6,
             "best recall@10": diskann_report["recall@10"].max(),
             "ms/query at best": diskann_report.loc[diskann_report["recall@10"].idxmax(), "ms/query"]},
        ]
    ).set_index("index")
    display(summary.round(3))

    # Put HNSW back so the following sections measure the same thing on every server.
    await session.rollback()
    async with engine.begin() as conn:
        await conn.run_sync(diskann.drop, checkfirst=True)
        await conn.run_sync(hnsw.create, checkfirst=True)
    print("restored", hnsw.name)

At 20k rows everything fits in memory, so DiskANN's main selling point (graceful behaviour when the
index spills to disk) does not show; what does show is the size difference from quantization and
the cost of the rescore round trip. On a table of millions of rows with a small `shared_buffers`,
run the same cells and the picture changes. Note also that filtering behaves differently: pgvectorscale
supports label-based filtering (`diskann` with a `smallint[]` labels column) rather than pgvector's
iterative scans.

### PRISM (pg_vectorsearch): partitions instead of a graph

[pg_vectorsearch](https://github.com/timescale/pg_vectorsearch) (Timescale, v0.1.0, alpha) adds a
third index type, `prism`, which is not a graph at all:

- **IVF-style partitions**: k-means splits the table into `nlist` lists, with a small routing tree
  over their centroids; a query scans only the `prism.nprobe` closest lists.
- **RaBitQ codes** in the lists (a few bits per dimension, with a per-vector error bound), so a probe
  reads compressed vectors, and a **rerank** step re-reads the full vectors from the heap only for
  candidates whose bound says they could still make the top-k.
- **One query knob**: `prism.nprobe` (`0` = auto, aiming at about 0.95 recall). The build sizes
  itself from the row count at `CREATE INDEX` time, so build it *after* loading.

It needs Postgres 18, so it only exists on the `pg18` server (`docker/Dockerfile.pg18`). It indexes
pgvector's `vector` column as it is: the operator class is `vec32_cosine_ops` instead of
`vector_cosine_ops`, and the query, `ORDER BY embedding <=> :q`, does not change. On every other
server the cell below reports "not available" and the subsection is skipped.

In [ ]:
await session.rollback()
try:
    async with engine.begin() as conn:
        await conn.execute(text("CREATE EXTENSION IF NOT EXISTS pg_vectorsearch"))
    ver = await session.scalar(text("SELECT extversion FROM pg_extension WHERE extname = 'pg_vectorsearch'"))
    HAS_PRISM = True
    print("pg_vectorsearch", ver)
except Exception as e:  # noqa: BLE001
    HAS_PRISM = False
    await session.rollback()
    print("prism not available on this server:", str(e).splitlines()[0])

Same procedure as for DiskANN: drop HNSW so the planner has one ANN index, build PRISM on the same
column, and read back what it decided. `prism_index_settings()` reports every parameter with its
source (`auto`, `derived`, `default`, or `session` when a GUC overrides it).

In [ ]:
prism = Index(
    "ix_demo_docs_embedding_prism",
    DemoDoc.embedding,
    postgresql_using="prism",
    postgresql_ops={"embedding": "vec32_cosine_ops"},
)
if HAS_PRISM:
    print(str(__import__("sqlalchemy").schema.CreateIndex(prism).compile(dialect=DIALECT)).strip())
    await session.rollback()
    t0 = time.perf_counter()
    async with engine.begin() as conn:
        await conn.run_sync(hnsw.drop, checkfirst=True)
        await conn.run_sync(prism.create, checkfirst=True)
    prism_build_s = time.perf_counter() - t0
    prism_size = await session.scalar(select(func.pg_relation_size(prism.name)))
    print(f"built in {prism_build_s:.1f}s, index size {prism_size / 1e6:.1f} MB")
    settings_df = pd.DataFrame(
        (await session.execute(text(f"SELECT * FROM prism_index_settings('{prism.name}')"))).mappings().all()
    )
    display(settings_df[settings_df["name"].isin(["nlist", "nprobe", "target_entries", "nlevels", "distance_mode"])])
    await explain(knn)

In [ ]:
# Recall and latency across nprobe, same 30 queries and the same exact answers as before
async def top10_prism(vec, nprobe: int) -> set[int]:
    stmt = select(DemoDoc.id).order_by(DemoDoc.embedding.cosine_distance(vec)).limit(10)
    await session.execute(text("SET LOCAL enable_indexscan = on"))
    await session.execute(text(f"SET LOCAL prism.nprobe = {nprobe}"))
    return set((await session.execute(stmt)).scalars())


prism_report = pd.DataFrame()
if HAS_PRISM:
    exact_sets = [await top10(v.tolist(), use_index=False) for v in queries]
    await session.rollback()
    rows = []
    for nprobe in (0, 2, 5, 10, 20, 40, 80, 160):  # 0 = auto; 160 > nlist probes every list
        hits, t = 0, 0.0
        for v, exact in zip(queries, exact_sets):
            t0 = time.perf_counter()
            approx = await top10_prism(v.tolist(), nprobe)
            t += time.perf_counter() - t0
            hits += len(exact & approx)
        rows.append({"nprobe": nprobe or "auto", "recall@10": hits / (10 * len(queries)),
                     "ms/query": 1000 * t / len(queries)})
    await session.rollback()
    prism_report = pd.DataFrame(rows)
prism_report.round(3)

`auto` is calibrated for real embeddings, which cluster: on 20k random 64-d vectors it picks
`nprobe = 10` of 144 lists and finds only about 40% of the true top-10, because the true neighbours
of a query are spread over many lists. Recall climbs as `nprobe` covers more of them, and with
every list probed it should reach the exact answer, because RaBitQ's error bound tells the rerank
step which candidates are still in doubt. DiskANN's one-bit codes stall well below that on the
same data.

The vendor numbers (100M vectors, 6 ms with a quarter of HNSW's memory, builds 20x faster) are about
scale. At 20k rows the columns to read are build time and size: on the pg18 run used to write this,
PRISM built in 0.3 s into 1.8 MB, against 2.7 s and 11.4 MB for HNSW and 20 s and 9.1 MB for DiskANN,
and probing all 144 lists gave 0.98 recall at about the latency of HNSW at `ef_search = 400`.


In [ ]:
if HAS_PRISM:
    rows = [{"index": "hnsw", "build s": hnsw_build_s, "size MB": hnsw_size / 1e6,
             "best recall@10": hnsw_report["recall@10"].max(),
             "ms/query at best": hnsw_report.loc[hnsw_report["recall@10"].idxmax(), "ms/query index"]}]
    if HAS_DISKANN:
        rows.append({"index": "diskann", "build s": diskann_build_s, "size MB": diskann_size / 1e6,
                     "best recall@10": diskann_report["recall@10"].max(),
                     "ms/query at best": diskann_report.loc[diskann_report["recall@10"].idxmax(), "ms/query"]})
    rows.append({"index": "prism", "build s": prism_build_s, "size MB": prism_size / 1e6,
                 "best recall@10": prism_report["recall@10"].max(),
                 "ms/query at best": prism_report.loc[prism_report["recall@10"].idxmax(), "ms/query"]})
    display(pd.DataFrame(rows).set_index("index").round(3))

    await session.rollback()
    async with engine.begin() as conn:
        await conn.run_sync(prism.drop, checkfirst=True)
        await conn.run_sync(hnsw.create, checkfirst=True)
    print("restored", hnsw.name)

## 7. Many connections at once

Everything so far ran one query at a time on one connection, which measures latency. A service
gets many requests at once, and then the question is *throughput*: how many queries per second
does the server sustain with N clients hammering it, and what happens to latency as N grows.

The cell below is a small load generator (a notebook version of `src/examples/bench/concurrency.py`):

- one **dedicated engine per level** with `pool_size=level, max_overflow=0`, so exactly N
  connections exist, and the pool never queues;
- one **worker coroutine per connection** that runs the query in a tight loop until the deadline,
  recording each latency;
- **bound parameters**: the statement is compiled once with `bindparam("q", type_=Vector(DIM))` and
  each iteration only supplies a new random vector. That is also how you would write it in an app;
- `echo=False` on the benchmark engine, or the logging would dominate the timings.

Three workloads: the HNSW kNN (index-bound), the same query with the index disabled (CPU-bound:
each query scans all rows, so this shows how many cores the server actually gets), and a
single-row insert (write path and commit cost). Docker Desktop's VM caps the cores for the
container, so an N beyond the core count mostly adds queueing.

In [ ]:
knn_param = (
    select(DemoDoc.id)
    .order_by(DemoDoc.embedding.cosine_distance(bindparam("q", type_=Vector(DIM))))
    .limit(10)
)
insert_param = insert(DemoDoc).values(category="bench", embedding=bindparam("q", type_=Vector(DIM)))

QUERY_VECS = rng.standard_normal((256, DIM))
QUERY_VECS /= np.linalg.norm(QUERY_VECS, axis=1, keepdims=True)
QUERY_VECS = [v.tolist() for v in QUERY_VECS]

# name -> (statement, per-connection setup SQL)
WORKLOADS = {
    "knn hnsw": (knn_param, None),
    "knn exact": (knn_param, "SET enable_indexscan = off"),
    "insert 1 row": (insert_param, None),
}


async def worker(eng, stmt, setup_sql, deadline, latencies):
    async with eng.connect() as conn:
        if setup_sql:
            await conn.execute(text(setup_sql))  # session-level SET: lasts for this connection
        while time.perf_counter() < deadline:
            t0 = time.perf_counter()
            await conn.execute(stmt, {"q": random.choice(QUERY_VECS)})
            await conn.commit()  # one transaction per request, as a real service would
            latencies.append((time.perf_counter() - t0) * 1000)


async def run_level(name, level, seconds):
    stmt, setup_sql = WORKLOADS[name]
    eng = create_async_engine(engine.url, pool_size=level, max_overflow=0, echo=False)
    conns = [await eng.connect() for _ in range(level)]  # open everything before the clock starts
    for c in conns:
        await c.close()
    lat = []
    t0 = time.perf_counter()
    await asyncio.gather(*(worker(eng, stmt, setup_sql, t0 + seconds, lat) for _ in range(level)))
    elapsed = time.perf_counter() - t0
    await eng.dispose()
    return {
        "workload": name, "conns": level, "queries": len(lat),
        "qps": len(lat) / elapsed,
        "p50 ms": float(np.percentile(lat, 50)), "p95 ms": float(np.percentile(lat, 95)),
    }


await session.rollback()  # don't hold a transaction open while other connections write
bench = []
for name in WORKLOADS:
    for level in CONCURRENCY_LEVELS:
        bench.append(await run_level(name, level, BENCH_SECONDS))
        print(f"{name:<13} x{level:<3} {bench[-1]['qps']:8.0f} qps  p95 {bench[-1]['p95 ms']:6.1f} ms")
bench = pd.DataFrame(bench)

Read the table by rows: how much does throughput grow from 1 to N connections, and what does that
cost in p95? A perfectly scaling server multiplies `qps` by N with flat latency until it runs out of
cores; after that `qps` plateaus and p95 grows roughly linearly with N (everyone waits in line).

- `knn hnsw` scales until the CPU is saturated; each query is sub-millisecond of work, so the
  round trip and the commit are a good share of the cost.
- `knn exact` is pure CPU (one full scan per query), so its plateau is the clearest picture of how
  many cores the container really has.
- `insert 1 row` also has to update the HNSW index, and every commit is a WAL flush.

To compare servers, run the notebook once per `SERVER` and put the two tables side by side.

In [ ]:
bench.round(1).pivot(index="conns", columns="workload", values=["qps", "p95 ms"])

In [ ]:
# Throughput relative to a single connection: the "scaling factor" per workload
base = bench[bench.conns == bench.conns.min()].set_index("workload")["qps"]
bench.assign(speedup=(bench.qps / bench.workload.map(base)).round(2)).pivot(
    index="conns", columns="workload", values="speedup"
)

In [ ]:
# The insert workload added rows; drop them so the rest of the notebook sees the original table.
from sqlalchemy import delete

n = (await session.execute(delete(DemoDoc).where(DemoDoc.category == "bench"))).rowcount
await session.commit()
print("removed", n, "bench rows")

## 8. Exploring with pandas and numpy

Pull the vectors once and do the rest locally: pairwise similarity matrices, clustering, plots.
`np.array(column.tolist())` turns the column of lists into an `(n, dim)` matrix.

In [ ]:
df = await q(select(Toy.name, Toy.embedding).where(Toy.embedding.is_not(None)).order_by(Toy.id), show=False)
E = np.array(df["embedding"].tolist())
En = E / np.linalg.norm(E, axis=1, keepdims=True)
S = En @ En.T  # cosine similarity matrix, (n, n)
pd.DataFrame(S, index=df["name"], columns=df["name"]).round(2)

In [ ]:
# For each item, its nearest neighbour in the table (excluding itself)
S2 = S.copy()
np.fill_diagonal(S2, np.nan)
nearest = pd.DataFrame(S2, index=df["name"], columns=df["name"])
pd.DataFrame({"nearest": nearest.idxmax(axis=1), "cosine_sim": nearest.max(axis=1).round(3)})

## 9. Your turn

- Rebuild the HNSW index with `vector_l2_ops` and re-run `explain(knn)`: the cosine query goes back
  to a `Seq Scan`, because the opclass doesn't match.
- Change `DemoDoc.embedding` to `HALFVEC(DIM)` (`from pgvector.sqlalchemy import HALFVEC`) and
  compare the index build time and table size (`pg_total_relation_size`).
- Swap `cosine_distance` for `max_inner_product` in the recall cell. Since the vectors are unit
  length, the ranking should be identical.
- Run `SET LOCAL hnsw.ef_search = 10` and then a query with `LIMIT 50`: pgvector caps results at
  `ef_search` unless iterative scans are on.

`await reset_toys()` restores the toy rows at any time; re-run the `demo_docs` cell to rebuild that table.
Run the notebook once per server (`-p SERVER pg16` / `-p SERVER pgrust`) and compare the build time,
the `EXPLAIN` output, the recall table and the concurrency tables. Raise `BENCH_SECONDS` to 5 or
more for steadier numbers, and try `CONCURRENCY_LEVELS = [1, 2, 4, 8]` if you know the container
has few cores.

In [ ]:
# Cleanup: drop the two demo tables and close the session.
await session.rollback()
async with engine.begin() as conn:
    await conn.run_sync(ToyBase.metadata.drop_all)
await session.close()
print("dropped toy_items, demo_docs")